In [1]:
import torch
import json
import io
import numpy as np
import scipy.special as sp

import pickle as pkl
import zlib
import base64

/Users/aleksei/.local/share/virtualenvs/kutulu-_n6nfavE/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from hydra import initialize, compose
from omegaconf import OmegaConf

In [3]:
import sys
sys.path.insert(0, '../')

In [4]:
from src.envs.agents.ppo_agent import PPOAgent

from src.envs.kutulu_observer import KutuluClosestObserver, KutuluClosestExtObserver
from src.envs.kutulu_world import KutuluWorldEnv
from src.game.template import (
    CELL_WALL, DEFAULT_KUTULU_ACTIONS, EXTENDED_KUTULU_ACTIONS, PPOConvSolver, PPOConvExtSolver, DQNConvSolver, DQNSolver,
)
from src.game.template import MOVE_REL_POS, REL_POSITIONS
from src.envs.agent_validator import AgentValidator
from src.game.template import DEFAULT_KUTULU_ACTIONS, EXTENDED_KUTULU_ACTIONS, UnreachedPositionError
from src.envs.league.agent_description import AgentDescription
from src.envs.trainer import Trainer, WOOD_MAZES, BRONZE_MAZES

from experiments.run_experiment import get_agent_info

In [5]:
competitors = [
    get_agent_info('05abe073de06428e896fcd880c9f3eac', new_experiment=True, output_dir='../../output'),
    get_agent_info('20250622-045641', new_experiment=False, output_dir='../../output'),
]

In [6]:
competitors[0]['legacy_encoder'] = True

In [7]:
# 1 / 314
experiment = 'b1c35809c181481d90fab3e07e54bcf4'
# mode = 'ppo_conv_ext'

In [8]:
config_path = f'../../kutulu_artifacts/mlflow_artifacts/{experiment}/artifacts/hydra_config'
checkpoint_dir = f'../../kutulu_artifacts/mlflow_artifacts/{experiment}/artifacts/models/agent_0/final'

In [9]:
with initialize(version_base=None, config_path=config_path):
    cfg = compose(config_name="config")

In [10]:
info = OmegaConf.to_container(cfg.agent, resolve=True)
# del info['type']
info['checkpoint_dir'] = checkpoint_dir
info['train'] = False

In [11]:
# agent = PPOAgent(**info)

In [12]:
# agent.model.load_state_dict(torch.load(f"{checkpoint_dir}/model.pt"))

In [13]:
# agent.train = False

In [14]:
competitors += [info]

In [15]:
len(competitors)

3

In [16]:
def _play_round(agents_info, league_level=4, num_envs=1):
    assert len(agents_info) == 4
    trainer = Trainer(
        num_experiments=1, agents_info=agents_info, shuffle=True,
        league_level=league_level, verbose=False, seed=17,
        silent=True, num_envs=num_envs, only_train=False, use_tqdm=False,
    )
    result = trainer.play_single_rollout(only_eval=True)
    # return result
    scores = np.array([np.argwhere(~np.isnan(result[:,i])).max().item() for i in range(4)])
    return scores

In [17]:
n_exps = 10
for competitor in competitors:
    agents_info = [info, competitor, competitor, competitor]
    n_wins = 0
    for _ in range(10):
        scores = _play_round(agents_info)
        if scores[0] == np.max(scores):
            n_wins += 1
    print(competitor['type'], n_wins / n_exps)
    # print(scores)

ppo 0.3
qdn_conv 0.8
ppo 0.8


In [37]:
n_exps = 10
for competitor in competitors:
    agents_info = [info, competitor, competitor, competitor]
    n_wins = 0
    for _ in range(10):
        scores = _play_round(agents_info)
        if scores[0] == np.max(scores):
            n_wins += 1
    print(competitor['type'], n_wins / n_exps)
    # print(scores)

ppo 0.1
qdn_conv 0.0
ppo 0.6
ppo 0.5


In [38]:
n_exps = 10
for competitor in competitors:
    agents_info = [info, competitor, competitor, competitor]
    n_wins = 0
    for _ in range(10):
        scores = _play_round(agents_info)
        if scores[0] == np.max(scores):
            n_wins += 1
    print(competitor['type'], n_wins / n_exps)
    # print(scores)

ppo 0.4
qdn_conv 0.1
ppo 0.5
ppo 0.5


In [65]:
av = AgentValidator(EXTENDED_KUTULU_ACTIONS)
av_plan = AgentValidator(EXTENDED_KUTULU_ACTIONS, player_params=(100, 1, 0))

In [66]:
av.check_entity_nearby(agent, 'SLASHER', n_min=2, n_max=2, verbose=True)

answer: {1, 3}, action: 1, explorers: [], wanderers: []
action: True, std: 0.0
#########
#..#.#..#
#...S...#
#..#.#..#
#...0^..#
#..#.#..#
#.......#
#..#.#..#
#########


answer: {0, 2}, action: 0, explorers: [], wanderers: []
action: True, std: 0.0
#########
#..#.#..#
#.......#
#..#^#..#
#...0.S.#
#..#.#..#
#.......#
#..#.#..#
#########


answer: {1, 3}, action: 1, explorers: [], wanderers: []
action: True, std: 0.0
#########
#..#.#..#
#.......#
#..#.#..#
#...0^..#
#..#.#..#
#...S...#
#..#.#..#
#########


answer: {0, 2}, action: 0, explorers: [], wanderers: []
action: True, std: 0.0
#########
#..#.#..#
#.......#
#..#^#..#
#.S.0...#
#..#.#..#
#.......#
#..#.#..#
#########




([True, True, True, True],
 [np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0)],
 [1, 0, 1, 0])

In [67]:
av.check_entity_nearby(agent, 'SLASHER', n_min=3, n_max=3)

(np.float64(1.0), np.float64(0.0), 2, np.float64(0.0), np.float64(0.0))

In [68]:
av.check_entity_nearby(agent, 'EXPLORER', n_min=2, n_max=3)

(np.float64(0.875), np.float64(0.0), 1, np.float64(0.0), np.float64(0.0))

In [69]:
av_plan.check_entity_nearby(agent, 'EXPLORER', n_min=1, n_max=2)

(np.float64(1.0), np.float64(0.0), 3, np.float64(0.0), np.float64(0.0))

In [70]:
av_plan.check_entity_nearby(agent, 'EXPLORER', n_min=3, n_max=3)

(np.float64(0.75), np.float64(0.0), 4, np.float64(0.0), np.float64(0.0))

In [71]:
av.check_entity_nearby(agent, 'WANDERER', n_min=1, n_max=2)

(np.float64(1.0), np.float64(0.0), 1, np.float64(0.0), np.float64(0.0))

In [72]:
av.check_entity_nearby(agent, 'WANDERER', n_min=1, n_max=2, env_types=['corner'])

(np.float64(1.0), np.float64(0.0), 4, np.float64(0.0), np.float64(0.0))

In [73]:
av.check_entity_nearby(agent, 'WANDERER', n_min=1, n_max=2, env_types=['coridor'])

(np.float64(1.0), np.float64(0.0), 4, np.float64(0.0), np.float64(0.0))

In [74]:
agent.train = False

In [75]:
weights = {}
for k,v in agent.model.state_dict().items():
    weights[k] = v.detach().numpy()
    print(k, v.shape)

conv1.weight torch.Size([8, 22, 3, 3])
conv1.bias torch.Size([8])
bn1.weight torch.Size([8])
bn1.bias torch.Size([8])
bn1.running_mean torch.Size([8])
bn1.running_var torch.Size([8])
bn1.num_batches_tracked torch.Size([])
conv2.weight torch.Size([8, 8, 3, 3])
conv2.bias torch.Size([8])
bn2.weight torch.Size([8])
bn2.bias torch.Size([8])
bn2.running_mean torch.Size([8])
bn2.running_var torch.Size([8])
bn2.num_batches_tracked torch.Size([])
fc.weight torch.Size([16, 72])
fc.bias torch.Size([16])
actor.weight torch.Size([8, 16])
actor.bias torch.Size([8])
critic.weight torch.Size([1, 16])
critic.bias torch.Size([1])
terminator.weight torch.Size([1, 16])
terminator.bias torch.Size([1])
occupation_head.weight torch.Size([1, 16])
occupation_head.bias torch.Size([1])


In [76]:
env = KutuluWorldEnv('', '', 1, actions=EXTENDED_KUTULU_ACTIONS)
env.map = [
    '###########',
    '#.........#',
    '#.#.#.#.#.#',
    '#.........#',
    '#.#.#.#.#.#',
    '#.........#',
    '###########',
]
env.width = len(env.map[0])
env.height = len(env.map)

In [77]:
from tests.utils import calculate_entities

In [78]:
player_pos = (3, 3)
explorers = [(5, 3), (1, 3)]
wanderers = [(3, 1, 1), (3, 5, 1), (3, 2, 0)]

entities = calculate_entities(player_pos, explorers, wanderers)
agent.set_env(env)
env._set_entities(entities)
env._set_players(entities, set_ids=True)

state = agent.observer.get_state(0)
# test_data = agent.episode_buffer.encode_states([state], return_tensors=False)
tensor_data = agent.episode_buffer.state_encoder.encode_states([state], return_tensors=True)

In [79]:
info = {
    'width': env.width,
    'height': env.height,
    'lines': env.map,
}

# solver = PPOConvExtSolver(info, EXTENDED_KUTULU_ACTIONS, weights, size=agent.size)

In [80]:
USED_ACTIONS = EXTENDED_KUTULU_ACTIONS
checkpoint_data = weights
SIZE = agent.size

In [81]:
if mode == 'qlearning':
    solver = QlearningSolver(info, USED_ACTIONS, checkpoint_data)
elif mode == 'dqn_ext':
    solver = DQNSolver(info, USED_ACTIONS, checkpoint_data)
elif mode == 'dqn_by_kind':
    solver = DQNByKindSolver(info, USED_ACTIONS, checkpoint_data)
elif mode == 'dqn_conv':
    solver = DQNConvSolver(info, USED_ACTIONS, checkpoint_data, SIZE)
elif mode == 'ppo_conv':
    solver = PPOConvSolver(info, USED_ACTIONS, checkpoint_data, SIZE)
elif mode == 'ppo_conv_ext':
    solver = PPOConvExtSolver(info, USED_ACTIONS, checkpoint_data, SIZE)
else:
    raise ValueError(f'unknown mode: "{mode}"')

In [82]:
np_output = solver._calculate_output([e.to_dict() for e in env._get_entites(0)], player_pos)

In [83]:
np_output

array([0.08214988, 0.15806647, 0.10088759, 0.13212442, 0.12885563,
       0.13235033, 0.13670304, 0.12886265])

In [84]:
model_output = agent.model(tensor_data)['policy'].detach().cpu().numpy()

In [85]:
model_output

array([[0.08214986, 0.15806648, 0.1008876 , 0.13212441, 0.12885563,
        0.13235033, 0.13670303, 0.12886265]], dtype=float32)

In [86]:
# data2, data1 = zip(*weights.items())

# data1 = pkl.dumps(data1)
# data2 = pkl.dumps(data2)

In [87]:
data1 = []
data2 = []
for k, v in weights.items():
    if 'num_batches_tracked' in k:
        continue
    print(k, v.shape)
    data2.append(k)
    buffer = io.BytesIO()
    v = v.astype(np.float16)
    np.save(buffer, v)
    data1.append(buffer.getvalue())
    # data1.append(zlib.compress(buffer.getvalue(), level=9))

data1 = pkl.dumps(data1)
data2 = pkl.dumps(data2)

conv1.weight (8, 22, 3, 3)
conv1.bias (8,)
bn1.weight (8,)
bn1.bias (8,)
bn1.running_mean (8,)
bn1.running_var (8,)
conv2.weight (8, 8, 3, 3)
conv2.bias (8,)
bn2.weight (8,)
bn2.bias (8,)
bn2.running_mean (8,)
bn2.running_var (8,)
fc.weight (16, 72)
fc.bias (16,)
actor.weight (8, 16)
actor.bias (8,)
critic.weight (1, 16)
critic.bias (1,)
terminator.weight (1, 16)
terminator.bias (1,)
occupation_head.weight (1, 16)
occupation_head.bias (1,)


In [88]:
with open('../src/game/template.py') as f:
    lines = f.readlines()

In [89]:
mode

'ppo_conv_ext'

In [90]:
with open('../src/game/template_submit.py', 'w') as f:
    for line in lines:
        line = line.replace("b'data1data1data1'", str(base64.b64encode(zlib.compress(data1, level=9))))
        line = line.replace("b'data2data2data2'", str(base64.b64encode(zlib.compress(data2, level=9))))
        line = line.replace("mode = 'mode'", f"mode = '{mode}'")
        line = line.replace("USED_ACTIONS = DEFAULT_KUTULU_ACTIONS", "USED_ACTIONS = EXTENDED_KUTULU_ACTIONS")
        line = line.replace("SIZE = 3", f"SIZE = {agent.size}")
        line = line.replace("experiment: ''", f"experiment: '{experiment}'")
        f.write(line)

In [91]:
!ls -lh ../src/game/template_submit.py

-rw-rw-r-- 1 kutulu kutulu 62K Sep 22 20:12 ../src/game/template_submit.py


In [92]:
experiment

'4000fbae13ff49b8b1adbada7cba6839'